<a href="https://colab.research.google.com/github/Anshitha9286/ASSIGNMENT/blob/main/GEN_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Powered Paraphrasing Tool
# Step 0: Confirm GPU is active

In [3]:
!nvidia-smi

Tue Sep 29 07:25:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1: Install Java (for grammar checking)


In [5]:
!apt-get update -qq
!apt-get install -y default-jre -qq

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package at-spi2-common.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../00-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../01-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1.1build1_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1.1b

## Step 2: Verify GPU access in PyTorch and test the grammar checker


In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

import language_tool_python
tool = language_tool_python.LanguageTool("en-US")
print(tool.check("This are a test sentence."))

True
Tesla T4


[Match({'rule_id': 'THIS_NNS', 'message': 'The singular demonstrative pronoun ‘this’ does not agree with the plural verb ‘are’. Did you mean “these”?', 'replacements': ['These'], 'offset_in_context': 0, 'context': 'This are a test sentence.', 'offset': 0, 'error_length': 4, 'category': 'GRAMMAR', 'rule_issue_type': 'grammar', 'sentence': 'This are a test sentence.'}), Match({'rule_id': 'PLURAL_VERB_AFTER_THIS', 'message': 'The verb ‘are’ is plural. Did you mean: “is”? Did you use a verb instead of a noun?', 'replacements': ['is'], 'offset_in_context': 5, 'context': 'This are a test sentence.', 'offset': 5, 'error_length': 3, 'category': 'GRAMMAR', 'rule_issue_type': 'grammar', 'sentence': 'This are a test sentence.'})]


## Step 3: Load the paraphrasing model


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "humarin/chatgpt_paraphraser_on_T5_base"  # or try Vamsi/T5_Paraphrase_Paws
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to("cuda")

config.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Step 4: Test paraphrase generation on a simple sentence


In [8]:
sentence = "The quick brown fox jumps over the lazy dog."
inputs = tokenizer("paraphrase: " + sentence, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_length=128,
    num_beams=5,
    num_return_sequences=5,
    no_repeat_ngram_size=2,
)

for o in outputs:
    print(tokenizer.decode(o, skip_special_tokens=True))

The speedy brown fox leaps over the sluggish dog.
The swift brown fox leaps over the lethargic dog.
The swift brown fox leaps over the lethargic canine.
The hurried brown fox leaps over the sluggish dog.
The speedy brown fox leaps over the lethargic dog.


## Step 5: Test on a longer, more technical sentence


In [9]:
sentence2 = "Machine learning models require large amounts of labeled data to achieve high accuracy on real-world tasks."
inputs2 = tokenizer("paraphrase: " + sentence2, return_tensors="pt").to("cuda")
outputs2 = model.generate(**inputs2, max_length=128, num_beams=5, num_return_sequences=5, no_repeat_ngram_size=2)
for o in outputs2:
    print(tokenizer.decode(o, skip_special_tokens=True))

Large quantities of labeled data are necessary for machine learning models to perform tasks with high accuracy.
Large quantities of labeled data are necessary for machine learning models to function effectively on real-world tasks.
Machine learning models need large quantities of labeled data to perform tasks with high accuracy.
Large quantities of labeled data are necessary for machine learning models to be accurate in real-world applications.
Large quantities of labeled data are necessary for machine learning models to be accurate in real-world situations.


## Step 6: Split text into sentences


In [10]:
import spacy
nlp = spacy.load("en_core_web_sm")

def split_sentences(text):
    doc = nlp(text.strip())
    return [s.text.strip() for s in doc.sents if s.text.strip()]

paragraph = "Artificial intelligence is transforming the way businesses operate. Companies use machine learning to automate tasks and make better decisions."
sentences = split_sentences(paragraph)
print(sentences)

['Artificial intelligence is transforming the way businesses operate.', 'Companies use machine learning to automate tasks and make better decisions.']


## Step 7: Wrap paraphrase generation into a reusable function


In [11]:
def generate_candidates(sentence, n=5):
    inputs = tokenizer("paraphrase: " + sentence, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_length=128, num_beams=n, num_return_sequences=n, no_repeat_ngram_size=2)
    return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]

for s in sentences:
    print("ORIGINAL:", s)
    for c in generate_candidates(s):
        print("  -", c)
    print()

ORIGINAL: Artificial intelligence is transforming the way businesses operate.
  - The use of artificial intelligence is causing a shift in business practices.
  - The use of artificial intelligence is altering the workings of businesses.
  - The implementation of artificial intelligence is altering the business landscape.
  - The implementation of artificial intelligence is causing a shift in business practices.
  - The use of artificial intelligence is changing the way businesses operate.

ORIGINAL: Companies use machine learning to automate tasks and make better decisions.
  - The use of machine learning by companies is aimed at automating tasks and improving decision-making.
  - Automated tasks and better decision-making are the goals of companies using machine learning.
  - The use of machine learning by companies is aimed at automating tasks and improving decision-making capabilities.
  - Automated tasks and better decision-making are the goals of machine learning in corporate app

## Step 8: Grammar and spelling check


In [12]:
def grammar_check(text):
    matches = tool.check(text)
    return len(matches)

## Step 9: Fluency check (perplexity)


In [13]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import math

gpt_tok = GPT2TokenizerFast.from_pretrained("gpt2")
gpt_model = GPT2LMHeadModel.from_pretrained("gpt2").to("cuda").eval()

def perplexity(text):
    enc = gpt_tok(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        loss = gpt_model(**enc, labels=enc["input_ids"]).loss
    return math.exp(loss.item())

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Step 10: Meaning preservation check (semantic similarity)


In [14]:
from sentence_transformers import SentenceTransformer, util

sbert = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

def similarity(a, b):
    emb = sbert.encode([a, b], convert_to_tensor=True)
    return util.cos_sim(emb[0], emb[1]).item()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Step 11: Originality check


In [15]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
smooth = SmoothingFunction().method1

def originality(original, candidate):
    bleu = sentence_bleu([original.lower().split()], candidate.lower().split(), smoothing_function=smooth)
    return 1 - bleu

## Step 12: Test all four quality checks together


In [16]:
orig = "Artificial intelligence is transforming the way businesses operate."
cand = "The use of artificial intelligence is causing a shift in business practices."

print("grammar errors:", grammar_check(cand))
print("perplexity:", perplexity(cand))
print("similarity:", similarity(orig, cand))
print("originality:", originality(orig, cand))

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


grammar errors: 0
perplexity: 25.09763931098611
similarity: 0.849920392036438
originality: 0.9094124765841224


## Step 13: Sanity-check the metrics at the extremes


In [18]:
# Test 1: nonsense sentence should have much higher (worse) perplexity
nonsense = "Dog the artificial is transforming intelligence."
print("nonsense perplexity:", perplexity(nonsense))

# Test 2: identical sentence should have similarity ~1.0 and originality ~0
print("self similarity:", similarity(orig, orig))
print("self originality:", originality(orig, orig))

# Test 3: unrelated sentence should have low similarity
unrelated = "I bought a bag of apples at the market yesterday."
print("unrelated similarity:", similarity(orig, unrelated))

nonsense perplexity: 2672.912489442063
self similarity: 1.0
self originality: 0.0
unrelated similarity: 0.13355162739753723


## Step 14: Score a candidate (combine all four checks)


In [19]:
def score_candidate(original, candidate):
    sim = similarity(original, candidate)
    orig_score = originality(original, candidate)
    flu = 1 / (1 + perplexity(candidate) / 100)   # convert perplexity to a 0-1 scale
    g_err = grammar_check(candidate)
    combined = 0.5 * sim + 0.3 * orig_score + 0.2 * flu
    return {"text": candidate, "similarity": sim, "originality": orig_score,
            "fluency": flu, "grammar_errors": g_err, "combined": combined}

## Step 15: Filter and select the best candidate


In [20]:
MIN_SIM = 0.75

def select_best(original, candidates):
    scored = [score_candidate(original, c) for c in candidates if c.strip().lower() != original.strip().lower()]
    good = [s for s in scored if s["similarity"] >= MIN_SIM and s["grammar_errors"] == 0]
    pool = good if good else scored   # fallback if nothing passes
    return max(pool, key=lambda s: s["combined"])

## Step 16: Test the full scoring + selection process


In [21]:
sentence = "Artificial intelligence is transforming the way businesses operate."
candidates = generate_candidates(sentence)
best = select_best(sentence, candidates)

for c in candidates:
    print(score_candidate(sentence, c))
print("\nBEST:", best)

{'text': 'The use of artificial intelligence is causing a shift in business practices.', 'similarity': 0.849920392036438, 'originality': 0.9094124765841224, 'fluency': 0.7993755961405897, 'grammar_errors': 0, 'combined': 0.8576590582215736}
{'text': 'The use of artificial intelligence is altering the workings of businesses.', 'similarity': 0.8992418050765991, 'originality': 0.8997484256852186, 'fluency': 0.7054597893526386, 'grammar_errors': 0, 'combined': 0.8606373881143928}
{'text': 'The implementation of artificial intelligence is altering the business landscape.', 'similarity': 0.86693274974823, 'originality': 0.8877555535138854, 'fluency': 0.6455513154610855, 'grammar_errors': 0, 'combined': 0.8289033040204977}
{'text': 'The implementation of artificial intelligence is causing a shift in business practices.', 'similarity': 0.8424339294433594, 'originality': 0.9094124765841224, 'fluency': 0.7222611192952294, 'grammar_errors': 0, 'combined': 0.8384929315559623}
{'text': 'The use of 

## Step 17: Full paraphrasing pipeline


In [22]:
def paraphrase_text(text):
    sentences = split_sentences(text)
    output_sentences = []
    for s in sentences:
        candidates = generate_candidates(s)
        best = select_best(s, candidates)
        output_sentences.append(best["text"])
    result = " ".join(output_sentences)
    return {
        "original": text,
        "paraphrase": result,
        "similarity": similarity(text, result),
        "originality": originality(text, result),
        "grammar_errors": grammar_check(result),
    }

## Step 18: Console input/output


In [24]:
def main():
    print("=== AI Paraphrasing Tool ===")
    text = input("Enter text to paraphrase:\n> ").strip()
    if not text:
        print("No text entered.")
        return
    result = paraphrase_text(text)
    print("\nORIGINAL   :", result["original"])
    print("PARAPHRASED:", result["paraphrase"])
    print(f"\nSimilarity : {result['similarity']:.3f}")
    print(f"Originality: {result['originality']:.3f}")
    print(f"Grammar errors: {result['grammar_errors']}")

main()

=== AI Paraphrasing Tool ===
Enter text to paraphrase:
> > Artificial intelligence is transforming the way businesses operate. Companies use machine learning to automate tasks and make better decisions. This has led to significant improvements in efficiency.

ORIGINAL   : > Artificial intelligence is transforming the way businesses operate. Companies use machine learning to automate tasks and make better decisions. This has led to significant improvements in efficiency.
PARAPHRASED: The use of artificial intelligence is altering the workings of businesses. The use of machine learning by companies is aimed at automating tasks and improving decision-making. Efficiency has been greatly improved as a result of this.

Similarity : 0.939
Originality: 0.958
Grammar errors: 0


## Step 19: Sample test texts


In [25]:
test_texts = [
    "Artificial intelligence is transforming the way businesses operate. Companies use machine learning to automate tasks and make better decisions.",
    "Climate change is causing global temperatures to rise. This leads to melting glaciers and more extreme weather events.",
    "The study found that students who sleep eight hours perform better on exams. Lack of sleep reduces memory and concentration.",
    "Python is a popular programming language known for its simple syntax. It is widely used in data science and web development.",
    "I was really tired after work yesterday, so I just ordered pizza and watched a movie at home.",
    "The company reported a sharp increase in quarterly revenue. Management expects the growth to continue next year.",
    "Regular exercise improves heart health and lowers stress. Doctors recommend at least thirty minutes of activity each day.",
    "The Industrial Revolution began in Britain in the eighteenth century. It changed manufacturing, transport and everyday life.",
]

## Step 20: Run the tool on all sample texts


In [26]:
results = [paraphrase_text(t) for t in test_texts]
for r in results:
    print("ORIGINAL   :", r["original"])
    print("PARAPHRASED:", r["paraphrase"])
    print(f"sim={r['similarity']:.3f}  orig={r['originality']:.3f}  errors={r['grammar_errors']}")
    print("-" * 80)

ORIGINAL   : Artificial intelligence is transforming the way businesses operate. Companies use machine learning to automate tasks and make better decisions.
PARAPHRASED: The use of artificial intelligence is altering the workings of businesses. The use of machine learning by companies is aimed at automating tasks and improving decision-making.
sim=0.940  orig=0.942  errors=0
--------------------------------------------------------------------------------
ORIGINAL   : Climate change is causing global temperatures to rise. This leads to melting glaciers and more extreme weather events.
PARAPHRASED: Climate change is contributing to a rise in global temperatures. The consequence of this is the melting of glaciers and the onset of more severe weather conditions.
sim=0.914  orig=0.947  errors=0
--------------------------------------------------------------------------------
ORIGINAL   : The study found that students who sleep eight hours perform better on exams. Lack of sleep reduces memory

## Step 21: Build the evaluation report


In [27]:
import pandas as pd
df = pd.DataFrame(results)
report = df[["similarity", "originality", "grammar_errors"]].round(3)
report.index = [f"Text {i+1}" for i in range(len(report))]
print(report)
print("\nAverages:")
print(report.mean().round(3))

        similarity  originality  grammar_errors
Text 1       0.940        0.942               0
Text 2       0.914        0.947               0
Text 3       0.958        0.929               0
Text 4       0.940        0.833               0
Text 5       0.930        0.922               0
Text 6       0.794        0.975               0
Text 7       0.964        0.877               0
Text 8       0.958        0.924               0

Averages:
similarity        0.925
originality       0.919
grammar_errors    0.000
dtype: float64


## Step 22: Inspect the weakest-scoring case


In [28]:
print("ORIGINAL   :", results[5]["original"])
print("PARAPHRASED:", results[5]["paraphrase"])

ORIGINAL   : The company reported a sharp increase in quarterly revenue. Management expects the growth to continue next year.
PARAPHRASED: The company's revenue experienced a significant surge in the third quarter. The growth is expected to persist next year, according to management.


Evaluation Summary

Overall: avg similarity 0.925 · avg originality 0.919 · grammar errors 0/8

Strengths

Zero grammar errors across all domains (technical, casual, business, academic, science)
Strong rewording (0.919 originality) without losing coherence
Multi-metric scoring picks better candidates than any single metric alone

Limitations

Weakest case (Text 6): model added an unsupported detail — "quarterly" → "third quarter" — that similarity scores didn't catch
Sentence-level only; no cross-sentence restructuring
BLEU-based originality ≠ plagiarism protection
Perplexity/grammar checks are automatic proxies, not human judgment

Improvements

Add fact-consistency checks (flag unsupported numbers/entities)
Fine-tune on formal/business text
Add cross-sentence context
Add external plagiarism check